# SigAlg's `Operators.variance` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `Operators.variance` method in SigAlg is a method for computing *variances* of random variables and vectors, both unconditional and conditional versions. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.Operators.variance).

## Mathematical definition

Let $X:\Omega \to \mathbb{R}$ be a random variable on a probability space $(\Omega, \mathcal{F},P)$ for which $E(X^2) < \infty$, and let $\mathcal{G}$ be a sub-$\sigma$-algebra of $\mathcal{F}$. The *conditional variance* of $X$ with respect to $\mathcal{G}$ is any $\mathcal{G}$-measurable random variable $V(X \mid \mathcal{G})$ for which

$$
V(X\mid \mathcal{G}) = E\left[ (X-E(X\mid \mathcal{G}))^2 \mid \mathcal{G}\right].
$$

In the case that $\Omega$ is finite (as it always is, in SigAlg), the $\sigma$-algebra $\mathcal{G}$ is determined by its (finitely many) atoms, and the space $L^2(\Omega, \mathcal{G}, P)$ has an orthogonal basis given by the indicator functions of the atoms of $\mathcal{G}$ with nonzero probability. Then we have

$$
V(X\mid \mathcal{G}) = \sum_B V(X|_B) I_B,
$$

where the sum extends over all atoms $B$ of $\mathcal{G}$ with nonzero probability, and where $V(X|_B)$ is the variance of the restricted random variable $X|_B:B\to \mathbb{R}$ on $B$ equipped with the conditional probability measure $P_B$ with $P_B(C) = P(C)/P(B)$ for $C\subset B$.

If $X : \Omega \to \mathbb{R}^d$ is a random vector of dimension $d>1$, with components

$$
X = (X_1,X_2,\ldots,X_d),
$$

then this method returns a `RandomVector` whose component random variables are the conditional variances $V(X_j \mid \mathcal{G})$, for $j=1,2,\ldots,d$.

## API examples


### Unconditional variances

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [3]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=5)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.3,
        3: 0.25,
        4: 0.2,
    }
)

Define a random variable $X: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set its `probability_measure` attribute to $P$ so that all variances will be computed relative to $P$.

In [4]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
X.prob_measure = P

All variances in SigAlg are instances of `RandomVariable`. The unconditional variance `V(X)` is thus a constant random variable whose value is the usual variance $V(X) = E\left[(X-E(X))^2\right]$. The `item` method extracts $V(X)$ from `V(X)`.

In [5]:
from sigalg.core import Operators

V = Operators.variance

variance_rv = V(X)
variance = V(X).item()

print(variance_rv, "\n")
print(variance)

Random variable 'V(X)':
          V(X)
sample        
0       5.1875
1       5.1875
2       5.1875
3       5.1875
4       5.1875 

5.1875


### Conditional variances

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$ and $A_1 = \{2,3,4\}$. 

In [6]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
    }
)

Compute the conditional variance $V(X\mid \mathcal{G})$. Notice that $X$ is constant on the atom $A_0$ of $\mathcal{G}$, and hence its variance is $0$ on this atom.

In [7]:
print(V(X,G))

Random variable 'V(X|G)':
          V(X|G)
sample          
0       0.000000
1       0.000000
2       5.555556
3       5.555556
4       5.555556


### Testing properties of variances

#### Conditional variances are linear combinations

We noted in the definition that the conditional variance may be expressed as a linear combination of the indicator functions of the atoms of the $\sigma$-algebra. In the next code cell, we test this. Notice that the output matches the output above.

In [8]:
I = RandomVariable.indicator_of

linear_combo = sum([V(X(A)).item() * I(A) for A in G.to_atoms()])

print(linear_combo.with_name("linear_combo"))

Random variable 'linear_combo':
        linear_combo
sample              
0           0.000000
1           0.000000
2           5.555556
3           5.555556
4           5.555556


#### The law of total variance

The law of total variance says that

$$
V(X) = E(V(X\mid \mathcal{G})) + V(E(X\mid \mathcal{G}))
$$

We verify this equality in the following code cell.

In [9]:
E = Operators.expectation

variance = V(X)
rhs = E(V(X,G)) + V(E(X,G))

print(variance, "\n")
print(rhs)

Random variable 'V(X)':
          V(X)
sample        
0       5.1875
1       5.1875
2       5.1875
3       5.1875
4       5.1875 

Random variable '(E(V(X|G))+V(E(X|G)))':
        (E(V(X|G))+V(E(X|G)))
sample                       
0                      5.1875
1                      5.1875
2                      5.1875
3                      5.1875
4                      5.1875


#### The "short-cut formula" for variance

The "short-cut formula" for the variance says that

$$
V(X\mid \mathcal{G}) = E(X^2 \mid \mathcal{G}) - E(X \mid \mathcal{G})^2.
$$

We verify this equality in the following code cell:

In [10]:
variance = V(X, G)
short_cut = E(X**2, G) - E(X, G) ** 2

print(variance, "\n")
print(short_cut)

Random variable 'V(X|G)':
          V(X|G)
sample          
0       0.000000
1       0.000000
2       5.555556
3       5.555556
4       5.555556 

Random variable '(E((X**2)|G)-(E(X|G)**2))':
        (E((X**2)|G)-(E(X|G)**2))
sample                           
0                        0.000000
1                        0.000000
2                        5.555556
3                        5.555556
4                        5.555556


#### Complete information yields no variance

If $X$ is $\mathcal{H}$-measurable, then we must have $V(X \mid \mathcal{H})=0$. We verify this in the next cell:

In [11]:
X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
H = SigmaAlgebra(sample_space=Omega, name="H").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 2,
        4: 3,
    }
)

print(V(X,H))

Random variable 'V(X|H)':
        V(X|H)
sample        
0          0.0
1          0.0
2          0.0
3          0.0
4          0.0
